In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Load the dataset
X, y = load_wine(return_X_y=True)
wine = load_wine()

print('Wine dataset shape:', X.shape)
print('Classes:', wine.target_names)
print('Features:', wine.feature_names)

In [ ]:
# TODO: Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

In [ ]:
# TODO: Build a KNN pipeline
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  KNeighborsClassifier(n_neighbors=5))
])

# TODO: Build a Random Forest pipeline
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=100, random_state=42))
])

print('Pipelines created!')

In [ ]:
# TODO: Compare using cross_val_score
knn_cv = cross_val_score(knn_pipe, X_train, y_train, cv=5)
rf_cv  = cross_val_score(rf_pipe,  X_train, y_train, cv=5)

print(f'KNN          CV Accuracy: {knn_cv.mean():.3f} +/- {knn_cv.std():.3f}')
print(f'Random Forest CV Accuracy: {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}')

best_pipe = rf_pipe if rf_cv.mean() >= knn_cv.mean() else knn_pipe
best_name = 'Random Forest' if rf_cv.mean() >= knn_cv.mean() else 'KNN'
print(f'\nBetter model based on CV: {best_name}')

In [ ]:
# TODO: Train and test the best model
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(f'--- {best_name} — Final Test Results ---\n')
print(classification_report(y_test, y_pred, target_names=wine.target_names))

In [ ]:
# BONUS: Confusion matrix for the best model
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Greens')
plt.colorbar(im)

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(wine.target_names, rotation=30, ha='right')
ax.set_yticklabels(wine.target_names)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix — {best_name} on Wine')

for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)

plt.tight_layout()
plt.show()